# Aplicação de Modelos de Previsão para Apoiar Decisões de Procurement visando a Redução de Custos

**Autora:** Izabella Santos  
**Orientadora:** Ana Paula Lopes  
**Instituição:** Instituto Superior de Contabilidade e Administração do Porto  
**Grau:** Mestrado em Logística  
**Ano:** 2026

---

## Descrição

Este notebook implementa o modelo de previsão de procura desenvolvido na dissertação de mestrado.  
São comparados cinco modelos aplicados à série temporal diária do *dataset* **DataCo Smart Supply Chain** (janeiro 2015 – setembro 2017):

| Modelo | Tipo |
|---|---|
| Naïve | Benchmark |
| SARIMA (1,0,1)(1,0,1)₇ | Estatístico |
| Holt-Winters (sazonalidade semanal) | Estatístico |
| Random Forest | Machine Learning |
| XGBoost | Machine Learning |

**Dataset:** [DataCo Smart Supply Chain — Kaggle](https://www.kaggle.com/datasets/shashwatwork/dataco-smart-supply-chain-for-big-data-analysis)  
Após descarregar, colocar o ficheiro CSV na pasta `DataBase/`.

---

## 0. Instalação de Dependências

Executar apenas se as bibliotecas não estiverem instaladas.

In [ ]:
# !pip install pandas numpy matplotlib seaborn statsmodels scikit-learn xgboost

## 1. Importações e Configuração Gráfica

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Modelos estatísticos
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Modelos de Machine Learning
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from xgboost import XGBRegressor

# Paleta de cores da dissertação
sns.set_style('whitegrid')
colors = {
    'treino': '#426871',
    'teste':  '#D2B04C',
    'naive':  '#7B7B7B',
    'arima':  '#A3623A',
    'hw':     '#C39B6A',
    'rf':     '#016E51',
    'xgb':    '#1A3C5E'
}

print('Bibliotecas carregadas com sucesso.')

## 2. Carregamento e Preparação dos Dados

O *dataset* DataCo cobre o período 2015–2018. Os dados de outubro a dezembro de 2017 e janeiro de 2018 são excluídos por apresentarem uma quebra abrupta de ~80% no volume, indicativa de dados incompletos.

In [ ]:
df = pd.read_csv("DataBase/DataCoSupplyChainDataset.csv", encoding="ISO-8859-1")
df['order_date'] = pd.to_datetime(df['order date (DateOrders)'])
df = df[(df['order_date'] >= '2015-01-01') & (df['order_date'] <= '2017-09-30')]

target = 'Order Item Quantity'

# Agregação diária
df_ts = df.groupby(df['order_date'].dt.date)[target].sum()
df_ts.index = pd.to_datetime(df_ts.index)
df_ts = df_ts.to_frame()
df_ts = df_ts.asfreq('D', fill_value=0)

print(f"Série total : {len(df_ts)} dias")
print(f"Período     : {df_ts.index[0].date()} → {df_ts.index[-1].date()}")
print(f"Média diária: {df_ts[target].mean():.1f} unidades")
print(f"Desvio-padrão: {df_ts[target].std():.1f} unidades")

## 3. Divisão Treino / Teste (80% / 20%)

In [ ]:
train_size = int(len(df_ts) * 0.8)
train = df_ts[:train_size]
test  = df_ts[train_size:]

print(f"Treino : {len(train)} dias ({train.index[0].date()} → {train.index[-1].date()})")
print(f"Teste  : {len(test)} dias ({test.index[0].date()} → {test.index[-1].date()})")

# Visualização da série completa
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(train.index, train[target], color=colors['treino'], label='Treino', linewidth=1)
ax.plot(test.index,  test[target],  color=colors['teste'],  label='Teste',  linewidth=1)
ax.axvline(x=test.index[0], color='grey', linestyle='--', alpha=0.5)
ax.set_title('Série Temporal Diária — Procura Agregada (2015–2017)', fontsize=13, weight='bold')
ax.set_xlabel('Data')
ax.set_ylabel('Quantidade')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Diagnóstico de Séries Temporais

### 4.1 Testes de Estacionaridade (ADF e KPSS)

Para a correcta parametrização do SARIMA é necessário determinar formalmente o parâmetro de diferenciação **d**:
- **Teste ADF**: H₀ = série não estacionária (raiz unitária). Rejeitar se *p* ≤ 0,05
- **Teste KPSS**: H₀ = série estacionária. Rejeitar se *p* < 0,05

In [ ]:
adf_result  = adfuller(train[target], autolag='AIC')
kpss_result = kpss(train[target], regression='c', nlags='auto')

print("=" * 55)
print("DIAGNÓSTICO DE ESTACIONARIDADE")
print("=" * 55)
print(f"\n[ADF]  Estatística: {adf_result[0]:.4f} | p-value: {adf_result[1]:.4f}")
print(f"[KPSS] Estatística: {kpss_result[0]:.4f} | p-value: {kpss_result[1]:.4f}")

adf_stat  = adf_result[1]  <= 0.05
kpss_stat = kpss_result[1] >= 0.05

if adf_stat and kpss_stat:
    d_order = 0
    print("\n→ Ambos os testes concordam: série estacionária. d = 0")
elif not adf_stat and not kpss_stat:
    d_order = 1
    print("\n→ Ambos os testes concordam: série não estacionária. d = 1")
else:
    d_order = 1
    print("\n→ Resultados contraditórios. Decisão conservadora: d = 1")

print(f"\n→ Parâmetro d definido formalmente: d = {d_order}")

### 4.2 Funções de Autocorrelação (ACF e PACF)

- **ACF** (*Autocorrelation Function*): identifica a ordem MA (*q*)
- **PACF** (*Partial Autocorrelation Function*): identifica a ordem AR (*p*)

In [ ]:
max_lags = min(30, len(train) // 2 - 1)
print(f"Lags calculados: {max_lags} (50% da amostra de treino)")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("ACF e PACF — Diagnóstico para Parametrização do SARIMA", fontsize=13, weight='bold')
plot_acf( train[target], lags=max_lags, ax=axes[0], color=colors['arima'])
plot_pacf(train[target], lags=max_lags, ax=axes[1], color=colors['treino'])
axes[0].set_title("ACF — Identifica ordem MA (q)")
axes[1].set_title("PACF — Identifica ordem AR (p)")
axes[0].set_xlabel("Lags (dias)")
axes[1].set_xlabel("Lags (dias)")
plt.tight_layout()
plt.show()

## 5. Função de Avaliação de Métricas

Os modelos são avaliados com base em duas métricas (Hyndman & Athanasopoulos, 2021):
- **RMSE** (*Root Mean Square Error*): penaliza erros de maior magnitude
- **MAPE** (*Mean Absolute Percentage Error*): interpretável em termos percentuais

In [ ]:
def avaliar_modelo(model_name, real, pred):
    rmse = np.sqrt(mean_squared_error(real, pred))
    mape = mean_absolute_percentage_error(real, pred)
    print(f"--- {model_name} ---")
    print(f"  RMSE : {rmse:.2f}")
    print(f"  MAPE : {mape:.2%}\n")
    return rmse, mape

## 6. Modelo Naïve — Benchmark

O modelo Naïve projecta o último valor observado para todos os períodos futuros.  
Serve como referência mínima: qualquer modelo que não supere o Naïve não justifica a sua complexidade adicional (Hyndman & Athanasopoulos, 2021).

In [ ]:
naive_fc = pd.Series(
    [train[target].iloc[-1]] * len(test),
    index=test.index
)
naive_rmse, naive_mape = avaliar_modelo("Naïve (Benchmark)", test[target], naive_fc)

## 7. Modelo SARIMA

O SARIMA (*Seasonal Auto-Regressive Integrated Moving Average*) estende o ARIMA com componentes sazonais P, D, Q e período sazonal *s*.  
Com dados diários, é configurado com sazonalidade semanal (*s* = 7) para capturar os padrões recorrentes entre dias da semana.  
Ordem seleccionada: **(1, d, 1)(1, 0, 1)₇**

In [ ]:
sarima_model = SARIMAX(
    train[target],
    order=(1, d_order, 1),
    seasonal_order=(1, 0, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

arima_fc = sarima_model.forecast(steps=len(test))
arima_fc.index = test.index
arima_rmse, arima_mape = avaliar_modelo("SARIMA (1,d,1)(1,0,1)₇", test[target], arima_fc)

## 8. Modelo Holt-Winters

O Holt-Winters (suavização exponencial tripla) captura tendência e sazonalidade.  
Configurado com componentes aditivas e sazonalidade semanal (*seasonal_periods* = 7).

In [ ]:
hw_model = ExponentialSmoothing(
    train[target],
    trend='add',
    seasonal='add',
    seasonal_periods=7,
    initialization_method="estimated"
).fit()

hw_fc = hw_model.forecast(len(test))
hw_fc.index = test.index
hw_rmse, hw_mape = avaliar_modelo("Holt-Winters (sazonal semanal)", test[target], hw_fc)

## 9. Feature Engineering para Modelos de Machine Learning

Para os modelos de ML são criadas 9 variáveis preditivas que codificam a estrutura temporal da série:

| Feature | Descrição |
|---|---|
| `day_num` | Tendência global (índice sequencial) |
| `day_of_week` | Sazonalidade semanal (0 = segunda … 6 = domingo) |
| `day_of_month` | Posição no mês |
| `month_of_year` | Sazonalidade anual |
| `lag_1` | Procura do dia anterior |
| `lag_7` | Procura do mesmo dia na semana anterior |
| `lag_14` | Procura do mesmo dia há duas semanas |
| `rolling_mean_7` | Média móvel dos últimos 7 dias |
| `rolling_mean_14` | Média móvel dos últimos 14 dias |

In [ ]:
df_feat = df_ts.copy()
df_feat['day_num']         = np.arange(len(df_feat))
df_feat['day_of_week']     = df_feat.index.dayofweek
df_feat['day_of_month']    = df_feat.index.day
df_feat['month_of_year']   = df_feat.index.month
df_feat['lag_1']           = df_feat[target].shift(1)
df_feat['lag_7']           = df_feat[target].shift(7)
df_feat['lag_14']          = df_feat[target].shift(14)
df_feat['rolling_mean_7']  = df_feat[target].rolling(7).mean()
df_feat['rolling_mean_14'] = df_feat[target].rolling(14).mean()
df_feat = df_feat.dropna()

features = ['day_num','day_of_week','day_of_month','month_of_year',
            'lag_1','lag_7','lag_14','rolling_mean_7','rolling_mean_14']

train_feat = df_feat[df_feat.index < test.index[0]]
test_feat  = df_feat[df_feat.index >= test.index[0]]

print(f"Observações de treino (após feature engineering): {len(train_feat)}")
print(f"Observações de teste : {len(test_feat)}")

## 10. Modelo Random Forest

O Random Forest combina múltiplas árvores de decisão independentes, minimizando o erro de variância.  
Configurado com 200 estimadores (árvores) e semente aleatória fixada para reprodutibilidade.

In [ ]:
rf_model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_model.fit(train_feat[features], train_feat[target])
rf_fc = pd.Series(rf_model.predict(test_feat[features]), index=test_feat.index)
rf_rmse, rf_mape = avaliar_modelo("Random Forest", test[target], rf_fc)

# Importância das features
feat_imp = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(9, 5))
feat_imp.plot(kind='barh', ax=ax, color=colors['rf'])
ax.set_title("Random Forest — Importância das Features", fontsize=13, weight='bold')
ax.set_xlabel("Importância relativa")
plt.tight_layout()
plt.show()

## 11. Modelo XGBoost

O XGBoost (*Extreme Gradient Boosting*) constrói árvores sequencialmente, onde cada árvore corrige os erros da anterior.  
Referência: Chen & Guestrin (2016). *XGBoost: A scalable tree boosting system*. KDD '16.

In [ ]:
xgb_model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)
xgb_model.fit(train_feat[features], train_feat[target])
xgb_fc = pd.Series(xgb_model.predict(test_feat[features]), index=test_feat.index)
xgb_rmse, xgb_mape = avaliar_modelo("XGBoost", test[target], xgb_fc)

# Importância das features
xgb_feat_imp = pd.Series(xgb_model.feature_importances_, index=features).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(9, 5))
xgb_feat_imp.plot(kind='barh', ax=ax, color=colors['xgb'])
ax.set_title("XGBoost — Importância das Features", fontsize=13, weight='bold')
ax.set_xlabel("Importância relativa")
plt.tight_layout()
plt.show()

## 12. Tabela Comparativa de Resultados

Ordenação por MAPE crescente (melhor modelo no topo).

In [ ]:
resultados = pd.DataFrame({
    'Modelo': ['Naïve (Benchmark)', 'SARIMA', 'Holt-Winters', 'Random Forest', 'XGBoost'],
    'Tipo':   ['Benchmark', 'Estatístico', 'Estatístico', 'Machine Learning', 'Machine Learning'],
    'RMSE':   [naive_rmse, arima_rmse, hw_rmse, rf_rmse, xgb_rmse],
    'MAPE':   [naive_mape, arima_mape, hw_mape, rf_mape, xgb_mape]
}).sort_values('MAPE').reset_index(drop=True)

resultados['MAPE (%)'] = resultados['MAPE'].map('{:.2%}'.format)
resultados['RMSE']     = resultados['RMSE'].map('{:.2f}'.format)
resultados.index = resultados.index + 1
resultados.index.name = 'Rank'

print(resultados[['Modelo','Tipo','RMSE','MAPE (%)']].to_string())
print(f"\n→ Melhor modelo: {resultados.iloc[0]['Modelo']}")

## 13. Gráfico Comparativo — RMSE e MAPE

In [ ]:
res = pd.DataFrame({
    'Modelo': ['Naïve', 'SARIMA', 'Holt-Winters', 'Random Forest', 'XGBoost'],
    'RMSE':   [naive_rmse, arima_rmse, hw_rmse, rf_rmse, xgb_rmse],
    'MAPE':   [naive_mape, arima_mape, hw_mape, rf_mape, xgb_mape]
}).sort_values('MAPE')

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(res))
w = 0.35
ax.bar(x - w/2, res['RMSE'], w, label='RMSE', color=colors['treino'])
ax2 = ax.twinx()
ax2.bar(x + w/2, res['MAPE']*100, w, label='MAPE (%)', color=colors['teste'], alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(res['Modelo'], rotation=15, ha='right')
ax.set_ylabel('RMSE')
ax2.set_ylabel('MAPE (%)')
ax.set_title('Comparação de Modelos — RMSE e MAPE (Dados Diários)', fontsize=13, weight='bold')
l1, lb1 = ax.get_legend_handles_labels()
l2, lb2 = ax2.get_legend_handles_labels()
ax.legend(l1+l2, lb1+lb2, loc='upper left')
plt.tight_layout()
plt.show()

## 14. Gráficos de Previsão por Modelo

Cada gráfico mostra os últimos 60 dias do período de treino e o período de teste completo (201 dias).

In [ ]:
def plot_forecast(train, test, forecast, model_name, color):
    fig, ax = plt.subplots(figsize=(14, 5))
    train_tail = train.tail(60)
    ax.plot(train_tail.index, train_tail[target],
            label='Histórico (Treino)', color=colors['treino'], linewidth=1.5)
    ax.plot(test.index, test[target],
            label='Real (Teste)', color=colors['teste'], linewidth=1.5)
    ax.plot(test.index, forecast,
            label=f'Previsão {model_name}', linestyle='--', color=color, linewidth=2)
    ax.set_title(f"Forecasting de Procura Diária (2015–2017) — {model_name}",
                 fontsize=13, weight='bold')
    ax.set_xlabel('Data')
    ax.set_ylabel('Quantidade')
    ax.legend(loc='upper left')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_forecast(train, test, naive_fc,  'Naïve (Benchmark)', colors['naive'])

In [ ]:
plot_forecast(train, test, arima_fc,  'SARIMA',            colors['arima'])

In [ ]:
plot_forecast(train, test, hw_fc,     'Holt-Winters',      colors['hw'])

In [ ]:
plot_forecast(train, test, rf_fc,     'Random Forest',     colors['rf'])

In [ ]:
plot_forecast(train, test, xgb_fc,    'XGBoost',           colors['xgb'])

---

## Referências

- Chen, T., & Guestrin, C. (2016). XGBoost: A scalable tree boosting system. *Proceedings of the 22nd ACM SIGKDD*, 785–794. https://doi.org/10.1145/2939672.2939785
- Chopra, S., & Meindl, P. (2016). *Supply chain management* (6.ª ed.). Pearson.
- Christopher, M. (2016). *Logistics & supply chain management* (5.ª ed.). Pearson.
- Hyndman, R. J., & Athanasopoulos, G. (2021). *Forecasting: Principles and practice* (3.ª ed.). OTexts. https://otexts.com/fpp3/
- Makridakis, S., Spiliotis, E., & Assimakopoulos, V. (2018). Statistical and Machine Learning forecasting methods. *PLoS ONE*, 13(3), e0194889. https://doi.org/10.1371/journal.pone.0194889
- Silver, E. A., Pyke, D. F., & Peterson, R. (1998). *Inventory management and production planning and scheduling* (3.ª ed.). Wiley.